# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fayrouzhassan2000/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages tend to be longer, younger, and slightly better positioned in search than declining pages. This is presented as an observational comparison between rising and falling content.

**Methodology question:**
How is the rising/declining label operationalized, and is the outcome period kept separate from the information used to characterize the pages?

**Why this matters:**
If information from the outcome period is also used to define or characterize the groups, the observed differences could be overstated. Keeping the outcome period separate makes the comparison easier to interpret as an observed relationship rather than evidence of causation.

### Finding 2 — The Content Performance Curve

The paper reports that content performance varies across age bands, with performance peaking around 61–90 days and declining substantially around 271–365 days.

**Methodology question:**
Does the validation design account for the temporal nature of content age, and would the observed pattern remain under a future-facing, time-aware evaluation?

**Why this matters:**
Content performance changes over time, so mixing observations from different lifecycle stages or time periods may affect the observed relationship. A time-aware evaluation would provide stronger evidence about whether the pattern is useful for future-facing decisions.

### Reflection

These questions are intended as constructive methodology checks rather than challenges to the findings. The goal is to understand how the study design supports the reported observations and what claims the evidence can reasonably support.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I evaluated the Random Forest model using a grouped train/test split with `client_hash_id` as the grouping variable. This was chosen to prevent pages from the same client from appearing in both the training and test sets, which could otherwise make the evaluation overly optimistic.

The Week-5 evaluation produced a **Precision@20 of 0.8** for the Random Forest, compared with **0.55** for the baseline.

For this validation audit, I re-run the same Week-5 model under the grouped split and compare the resulting Precision@20 with the previously measured value. This is a re-evaluation of an already honest validation design rather than a correction from a random split.

| Evaluation                         | Random Forest Precision@20 |
| ---------------------------------- | -------------------------: |
| Week 5 — grouped split             |                       0.8 |
| Week 6 — re-run with grouped split |           *measured below* |

The comparison is used to assess whether the observed model performance is consistent under the grouped validation setup. It is not treated as evidence that the model will achieve the same performance on future unseen data.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Data loading

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

from datasets import load_dataset

content_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

content_df = content_ds.to_pandas()

content_df = content_df[
    [
        "client_hash_id",
        "content_hash_id",
        "content_type"
    ]
]

feature_df = features.merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

march = feature_df.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

march["is_declining_label"] = (
    march["april_impressions"] < march["gsc_impressions"]
)


march.drop(columns=["april_impressions"], axis = 1, inplace=True)

df = march.copy()
# =========================
# Handling Missing Values
# =========================

# 1. GA4 Sessions
# NaN means the client has no GA4 access
df["ga4_sessions"] = df["ga4_sessions"].fillna(0)


# 2. GSC Average Position
# NaN means there were no GSC impressions
# Create an indicator before imputation
df["gsc_avg_position_missing"] = (
    df["gsc_avg_position"].isna().astype(int)
)

# Impute missing positions with the median
gsc_position_median = df["gsc_avg_position"].median()

df["gsc_avg_position"] = df["gsc_avg_position"].fillna(
    gsc_position_median
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
# Section 2 — Re-run Week-5 model under an honest grouped split

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

# 1. Define features and target
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "content_type",
    "gsc_avg_position_missing"
]

X = df[feature_cols]
y = df["is_declining_label"]

# 2. Use client as the grouping variable
groups = df["client_hash_id"]

# 3. Create grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# 4. Define numeric and categorical features
numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_avg_position_missing"
]

categorical_features = [
    "content_type"
]

# 5. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# 6. Build Random Forest pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(random_state=42)
        )
    ]
)

# 7. Train the model
rf_pipeline.fit(X_train, y_train)

# 8. Get probability scores for the declining class
rf_scores = rf_pipeline.predict_proba(X_test)[:, 1]

# 9. Rank test examples by predicted probability
K = 20

top_k_idx = np.argsort(rf_scores)[::-1][:K]

# 10. Calculate Precision@K
precision_at_k_w6 = y_test.iloc[top_k_idx].mean()

print(f"Week 5 Precision@{K}: 0.8")
print(f"Week 6 Precision@{K}: {precision_at_k_w6:.3f}")

Week 5 Precision@20: 0.8
Week 6 Precision@20: 0.800


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final feature set for information that would not be available at prediction time.

The prediction setup uses March 2026 information to identify whether content will decline in April 2026. Therefore, April performance information must not be used as a model feature.

| Feature                    | Source                  | Leakage risk           |
| -------------------------- | ----------------------- | ---------------------- |
| `gsc_impressions`          | March 2026              | No                     |
| `gsc_clicks`               | March 2026              | No                     |
| `gsc_avg_position`         | March 2026              | No                     |
| `ga4_sessions`             | March 2026              | No                     |
| `content_type`             | Content metadata        | No                     |
| `gsc_avg_position_missing` | Derived from March data | No                     |
| `april_impressions`        | April 2026 outcome      | Excluded from features |

`april_impressions` is used only to create the target:

`is_declining_label = april_impressions < march_impressions`

After the label is created, `april_impressions` is removed and is not provided to the model.

I also checked the role of `client_hash_id`. It is used only as the grouping variable for the train/test split and is not included in the final feature set. This keeps the same client from appearing in both training and test data while preventing client identity from becoming a predictive feature.

### Preprocessing leakage check

The original preprocessing calculated the median of `gsc_avg_position` before the train/test split. Although this did not use the target or April information, the test set could still influence the imputation value.

For this audit, I moved the imputation step inside the preprocessing pipeline. The median is therefore learned from the training data only and then applied to the test data.

This makes the validation pipeline more faithful to the information that would be available at prediction time.

Overall, I did not identify direct target leakage in the final feature set. The April outcome is kept separate from the predictors, the client identifier is used only for grouping, and preprocessing is fitted within the training pipeline.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


df = march.copy()
# =========================
# Handling Missing Values
# =========================

# 1. GA4 Sessions
# NaN means the client has no GA4 access
df["ga4_sessions"] = df["ga4_sessions"].fillna(0)


# 2. GSC
# NaN means there were no GSC impressions
# Create an indicator before imputation
df["gsc_avg_position_missing"] = (
    df["gsc_avg_position"].isna().astype(int)
)



In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "content_type",
    "gsc_avg_position_missing"
]

X = df[feature_cols]
y = df["is_declining_label"]

# Reuse the grouped train/test split
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_avg_position_missing"
]

categorical_features = [
    "content_type"
]

# Numeric preprocessing
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Full pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42))
    ]
)

# Train
rf_pipeline.fit(X_train, y_train)

# Predict probabilities
K = 20

rf_scores = rf_pipeline.predict_proba(X_test)[:, 1]

# Rank by highest probability of decline
top_k_idx = np.argsort(rf_scores)[::-1][:K]

# Precision@20
precision_at_k_w6 = y_test.iloc[top_k_idx].mean()

print(f"Random Forest Precision@{K}: {precision_at_k_w6:.3f}")

Random Forest Precision@20: 0.900


"After moving imputation inside the pipeline so that the median was learned from the training data only, the measured Precision@20 changed from 0.8 to 0.9 . This change reflects a different preprocessing procedure and should not be interpreted as evidence that leakage correction improved model performance."

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

> The Random Forest model is better at identifying declining content than the baseline.

### Safer claim

> In this evaluation, the Random Forest achieved a measured Precision@20 of 0.8 compared with 0.55 for the baseline under the client-grouped split. This is an observed result on this dataset and evaluation setup, and it provides directional evidence that the model may be useful for prioritizing potentially declining content. It should be treated as decision-support rather than evidence that the model will perform the same way on unseen clients or future data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.